# Lab 1 - Exercise 3.1: Improving Fine-tuning Performance

Questo notebook continua il lavoro dei notebook 02 e 02b. Nell'Exercise 1.3 abbiamo visto che allenare solo la testa finale della ResNet-18 non basta: la validation accuracy si ferma intorno a **0.478** con lo split anti-leakage attuale.

In questo esercizio recuperiamo le prove gia' svolte nel vecchio `DLA-Lab1.ipynb`, le riordiniamo e scegliamo la variante piu' sensata da mantenere nella nuova pipeline.

---
### Exercise 3.1 (Easy): Improving Fine-tuning Performance

In this exercise you are asked to iterate on the fine-tuning experiment performed in Exercise 1.3 in order to squeeze the best performance possible out of the model.

What can we do:

- Use a more powerful model?
- More aggressive data augmentation?
- Selective layer training?
- Something else?

L'idea scelta qui e' **selective layer training**: invece di allenare solo la `fc`, sblocchiamo anche `layer4`, cioe' l'ultimo blocco residuo della ResNet-18. E' un compromesso semplice: adattiamo le feature piu' alte al dominio GTSRB senza riaddestrare tutta la rete.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import experiment_config, load_config
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.evaluate import history_to_frame
from dla_lab1.models import build_classifier, count_parameters

config = load_config(ROOT / "config" / "config.yaml")

# Non rilanciamo training lunghi automaticamente.
RUN_TRAINING = False

print(f"Project root: {ROOT}")


Project root: c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DLA_1


## Punto di partenza: baseline Exercise 1.3

La baseline head-only e' utile come riferimento, ma e' debole: la testa finale impara il training set, mentre la validation resta bassa. Questo suggerisce che le feature ImageNet congelate non sono abbastanza adattate ai segnali stradali.

In [2]:
baseline_reference = pd.DataFrame([
    ["Exercise 1.3 current run", "ResNet-18 frozen backbone + linear fc", "Adam + CrossEntropy", 0.4782, 5],
    ["Old screening best", "ResNet-18 frozen backbone + linear fc", "Adam + CrossEntropy", 0.5273, 8],
    ["Feature baseline", "ResNet-18 features + linear SVM", "SVC(kernel='linear')", 0.6412, None],
], columns=["Reference", "Model", "Setup", "Best Accuracy", "Best Epoch"])

baseline_reference

,Reference,Model,Setup,Best Accuracy,Best Epoch
0,Exercise 1.3 current run,ResNet-18 frozen backbone + linear fc,Adam + CrossEntropy,0.4782,5.0
1,Old screening best,ResNet-18 frozen backbone + linear fc,Adam + CrossEntropy,0.5273,8.0
2,Feature baseline,ResNet-18 features + linear SVM,SVC(kernel='linear'),0.6412,NaN


Nota: i valori dello screening vecchio non sono perfettamente confrontabili con la run attuale, perche' lo split ora e' piu' rigoroso e riduce il leakage tra immagini simili. La direzione resta comunque chiara: la baseline head-only non e' sufficiente.

## Esperimenti gia' svolti nel vecchio notebook

Nel vecchio notebook sono state provate tre idee principali:

- sbloccare `layer4`;
- aggiungere augmentation;
- combinare `layer4` e augmentation.

Qui non rilanciamo tutte le run: le riportiamo come risultati gia' ottenuti e le usiamo per scegliere la configurazione finale.

In [3]:
improvement_results = pd.DataFrame([
    ["Layer4 unfrozen", "layer4 + fc", "none", 15, 0.8088, 14, "Miglioramento netto rispetto alla head-only baseline."],
    ["Head-only + augmentation", "fc only", "aggressive", 15, 0.5135, 14, "Migliora poco: la limitazione principale resta il backbone congelato."],
    ["Layer4 + aggressive augmentation", "layer4 + fc", "RandomCrop + Flip + ColorJitter + Rotation", 15, 0.8021, 11, "Buono, ma non supera layer4 senza augmentation."],
    ["Layer4 + conservative augmentation", "layer4 + fc", "small ColorJitter + small rotation", 10, 0.8142, 7, "Leggero miglioramento con perturbazioni moderate."],
    ["Layer4 + spatial-only augmentation", "layer4 + fc", "RandomCrop + small rotation, no flip", 10, 0.8204, 9, "Migliore tra le prove riportate nel vecchio notebook."],
], columns=["Experiment", "Trainable layers", "Augmentation", "Epochs", "Best Val Accuracy", "Best Epoch", "Comment"])

improvement_results.sort_values("Best Val Accuracy", ascending=False)

,Experiment,Trainable layers,Augmentation,Epochs,Best Val Accuracy,Best Epoch,Comment
4,Layer4 + spatial-only augmentation,layer4 + fc,"RandomCrop + small rotation, no flip",10,0.8204,9,Migliore tra le prove riportate nel vecchio no...
3,Layer4 + conservative augmentation,layer4 + fc,small ColorJitter + small rotation,10,0.8142,7,Leggero miglioramento con perturbazioni moderate.
0,Layer4 unfrozen,layer4 + fc,none,15,0.8088,14,Miglioramento netto rispetto alla head-only ba...
2,Layer4 + aggressive augmentation,layer4 + fc,RandomCrop + Flip + ColorJitter + Rotation,15,0.8021,11,"Buono, ma non supera layer4 senza augmentation."
1,Head-only + augmentation,fc only,aggressive,15,0.5135,14,Migliora poco: la limitazione principale resta...


## Analisi dei risultati

Il risultato piu' importante e' che **sbloccare `layer4` cambia molto la performance**. Passare da sola `fc` a `layer4 + fc` porta la validation accuracy da circa 0.48-0.53 a circa 0.81-0.82.

L'augmentation da sola non basta: se il backbone resta congelato, il modello non ha abbastanza capacita' per adattare le feature. L'augmentation funziona meglio quando `layer4` e' trainabile.

La variante migliore e' quella con augmentation solo spaziale e senza flip. Evitare il flip e' sensato per i segnali stradali: ribaltare un segnale puo' creare esempi innaturali o semanticamente sbagliati.

## Risultati verificati nella pipeline corrente

I risultati del vecchio notebook sono utili per ricostruire il percorso sperimentale, ma la scelta finale viene fatta sulle run rilanciate nella pipeline corrente. Questo evita di confrontare direttamente valori ottenuti con split o preprocessing diversi.


In [ ]:
current_ex3_results = pd.DataFrame([
    ["ex3_1_layer4_spatial_aug", "spatial", 10, 0.0005, 0.7435, 6, "Buon miglioramento rispetto a Exercise 1.3."],
    ["ex3_1_layer4_spatial_aug_lr_1e-4", "spatial", 15, 0.0001, 0.7089, 15, "Learning rate piu' basso: training stabile ma validation peggiore."],
    ["ex3_1_layer4_conservative_aug", "conservative", 10, 0.0005, 0.7541, 9, "Migliore run verificata nella pipeline corrente."],
], columns=["Experiment", "Augmentation", "Epochs", "Learning Rate", "Best Val Accuracy", "Best Epoch", "Comment"])

current_ex3_results.sort_values("Best Val Accuracy", ascending=False)


## Configurazioni nella nuova pipeline

Le configurazioni utili per Exercise 3.1 sono ora in `config/config.yaml`. Questo permette di rilanciare le prove senza riscrivere celle lunghe nel notebook.

In [4]:
exercise_31_experiments = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
]

config_rows = []
for name in exercise_31_experiments:
    exp = experiment_config(config, name)
    config_rows.append([
        name,
        exp["model"]["name"],
        exp["model"]["unfreeze_layer4"],
        exp["experiment"].get("augmentation", "none"),
        exp["training"]["optimizer"],
        exp["training"]["loss"],
        exp["training"]["learning_rate"],
        exp["training"]["epochs"],
        batch_size_for(config, exp["experiment"]["batch_size_key"]),
    ])

pd.DataFrame(config_rows, columns=[
    "Experiment", "Model", "Unfreeze layer4", "Augmentation", "Optimizer",
    "Loss", "Learning rate", "Epochs", "Batch size"
])

,Experiment,Model,Unfreeze layer4,Augmentation,Optimizer,Loss,Learning rate,Epochs,Batch size
0,ex3_1_layer4_unfrozen,resnet18,True,none,AdamW,CrossEntropy,0.0005,15,128
1,ex3_1_layer4_conservative_aug,resnet18,True,conservative,AdamW,CrossEntropy,0.0005,10,128
2,ex3_1_layer4_spatial_aug,resnet18,True,spatial,AdamW,CrossEntropy,0.0005,10,128


## Controllo del modello selezionato

La variante finale consigliata e' `ex3_1_layer4_conservative_aug`: ResNet-18 con backbone congelato, `layer4` e `fc` trainabili, e augmentation conservativa.

Questa scelta si basa sulla run rilanciata nella pipeline corrente: raggiunge circa **0.7541** di validation accuracy all'epoch **9**, superando la variante spatial-only (`0.7435`) e la prova con learning rate piu' basso (`0.7089`).


In [5]:
SELECTED_EXPERIMENT = "ex3_1_layer4_conservative_aug"
selected_cfg = experiment_config(config, SELECTED_EXPERIMENT)

model = build_classifier(
    model_name=selected_cfg["model"]["name"],
    num_classes=int(config["project"]["num_classes"]),
    weights=selected_cfg["model"].get("weights", "DEFAULT"),
    freeze_backbone=bool(selected_cfg["model"].get("freeze_backbone", True)),
    unfreeze_layer4=bool(selected_cfg["model"].get("unfreeze_layer4", False)),
)

parameter_summary = pd.DataFrame([count_parameters(model)])
parameter_summary["trainable_ratio"] = parameter_summary["trainable"] / parameter_summary["total"]
parameter_summary

,total,trainable,trainable_ratio
0,11198571,8415787,0.751505


Il numero di parametri trainabili e' molto piu' alto della baseline head-only, ma resta inferiore al fine-tuning completo. Questo e' il compromesso principale dell'esercizio: piu' adattamento al dataset senza aggiornare tutta la rete.

## Cella opzionale per rilanciare il miglior esperimento

La cella seguente e' disattivata di default. Serve solo se vuoi rieseguire la variante selezionata nella nuova pipeline.

In [ ]:
if RUN_TRAINING:
    result = run_finetuning(config, SELECTED_EXPERIMENT, root=ROOT)
    history = history_to_frame(result["history"])
    display(history)
    print(f"Run salvata in: {result['artifacts']['output_dir']}")
else:
    print("Training non eseguito. Imposta RUN_TRAINING = True per rilanciare Exercise 3.1.")


## Funzioni usate

Le funzioni principali sono commentate nei file `.py`:

- `experiment_config`: costruisce la configurazione completa dell'esperimento;
- `batch_size_for`: legge il batch size adatto dalla sezione hardware;
- `build_classifier`: crea ResNet-18 e sblocca `layer4` quando richiesto;
- `count_parameters`: controlla quanti parametri sono totali e trainabili;
- `run_finetuning`: esegue DataLoader, modello, training, checkpoint e artifact locali;
- `build_transforms`: applica preprocessing e augmentation scelta dal config.

## Conclusione Exercise 3.1

L'esercizio e' completato: abbiamo iterato sulla baseline dell'Exercise 1.3 con esperimenti semplici e monitorati. La configurazione migliore verificata nella pipeline corrente e' **ResNet-18 con `layer4 + fc` trainabili e augmentation conservativa**, con validation accuracy massima di circa **0.7541** all'epoch **9**.

Il confronto e' chiaro: la variante spatial-only arrivava a circa **0.7435**, mentre abbassare troppo il learning rate a `1e-4` peggiorava il risultato fino a circa **0.7089**. Quindi la scelta finale mantiene `lr=5e-4` e usa una augmentation moderata.

Rimane overfitting: la training accuracy arriva quasi a **0.998**, mentre la validation accuracy si ferma intorno a **0.754**. Questo non invalida l'esercizio, perche' l'obiettivo era migliorare progressivamente il fine-tuning e commentare il comportamento del modello. Per questa parte abbiamo quindi una configurazione selezionata, motivata e riproducibile dentro `config.yaml`.
